### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

### RAG Pipeline

In [2]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [4]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [5]:
output = rag_pipeline("Can you suggest me some earphones?")

In [6]:
output

{'data_object': RAGGenerationResponse(answer='Sure—here are some earphone options available:\n\n- **Wireless Earbuds (Bluetooth 5.3, IPX7, 37H playback)** – *ID: B0B9FTVL58* (rating **4.2**). In-ear style with deep bass, smart touch, and an LED power display.\n- **Open-Ear Wireless Earbuds (Bluetooth 5.3, up to 60H with case, IPX7, earhooks)** – *ID: B0CBMPG524* (rating **4.4**). Comfortable open-ear fit so you can hear your surroundings; great for running/workouts.\n- **Bone Conduction Open-Ear Headphones (Bluetooth 5.3, IP56, mic, up to 8H)** – *ID: B0BNHVLF7G* (rating **4.0**). Lets sound come through your cheekbone; good for sports and outdoor use.\n- **Wireless Earbuds with Noise Cancelling Mic (Bluetooth 5.1, IPX7, 30H w/case)** – *ID: B0C6K1GQCF* (rating **4.3**). Touch controls, deep bass, and clear calls.\n- **Kids Over-Ear Wired Headphones (94dB limit, foldable, 3.5mm jack)** – *ID: B0C142QS8X* (rating **4.5**). Volume-limited for safer listening; no mic on the headphones.\n\

In [7]:
print(output["answer"])

Sure—here are some earphone options available:

- **Wireless Earbuds (Bluetooth 5.3, IPX7, 37H playback)** – *ID: B0B9FTVL58* (rating **4.2**). In-ear style with deep bass, smart touch, and an LED power display.
- **Open-Ear Wireless Earbuds (Bluetooth 5.3, up to 60H with case, IPX7, earhooks)** – *ID: B0CBMPG524* (rating **4.4**). Comfortable open-ear fit so you can hear your surroundings; great for running/workouts.
- **Bone Conduction Open-Ear Headphones (Bluetooth 5.3, IP56, mic, up to 8H)** – *ID: B0BNHVLF7G* (rating **4.0**). Lets sound come through your cheekbone; good for sports and outdoor use.
- **Wireless Earbuds with Noise Cancelling Mic (Bluetooth 5.1, IPX7, 30H w/case)** – *ID: B0C6K1GQCF* (rating **4.3**). Touch controls, deep bass, and clear calls.
- **Kids Over-Ear Wired Headphones (94dB limit, foldable, 3.5mm jack)** – *ID: B0C142QS8X* (rating **4.5**). Volume-limited for safer listening; no mic on the headphones.

If you tell me whether you want **in-ear vs open-ear 

### RAG Pipeline with Grounding Context

In [8]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [11]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [12]:
output = rag_pipeline("Can you suggest me some earphones?", top_k=10)

In [13]:
output

{'data_object': RAGGenerationResponse(answer='Here are some earphones you can choose from (all available in the shop right now):\n\n- **Open-ear / ambient listening (no in-ear canal, earhooks): B0CBMPG524**\n  - **Design:** Open-ear true wireless earbuds; rests on your ears without covering the ear canal.\n  - **Comfort/Security:** Silicone ear hooks and ergonomic design for secure fit during running, walking, and workouts.\n  - **Sound:** 16.2mm vibrating diaphragm driver for balanced, clear audio.\n  - **Awareness:** Lets you hear your surroundings while listening (helpful outdoors).\n  - **Battery:** ~11 hours per charge; up to **60 hours** with the portable charging case.\n  - **Water resistance:** **IPX7** (note: charging case is not waterproof).\n  - **Bluetooth:** **Bluetooth 5.3**.\n\n- **In-ear earbuds with deep bass + mics (IPX7): B0B9FTVL58**\n  - **Type:** Wireless earbuds with charging case.\n  - **Bluetooth:** **Bluetooth 5.3**.\n  - **Playback:** ~**37 hours** playback (

In [14]:
print(output["answer"])

Here are some earphones you can choose from (all available in the shop right now):

- **Open-ear / ambient listening (no in-ear canal, earhooks): B0CBMPG524**
  - **Design:** Open-ear true wireless earbuds; rests on your ears without covering the ear canal.
  - **Comfort/Security:** Silicone ear hooks and ergonomic design for secure fit during running, walking, and workouts.
  - **Sound:** 16.2mm vibrating diaphragm driver for balanced, clear audio.
  - **Awareness:** Lets you hear your surroundings while listening (helpful outdoors).
  - **Battery:** ~11 hours per charge; up to **60 hours** with the portable charging case.
  - **Water resistance:** **IPX7** (note: charging case is not waterproof).
  - **Bluetooth:** **Bluetooth 5.3**.

- **In-ear earbuds with deep bass + mics (IPX7): B0B9FTVL58**
  - **Type:** Wireless earbuds with charging case.
  - **Bluetooth:** **Bluetooth 5.3**.
  - **Playback:** ~**37 hours** playback (per description), with LED power display.
  - **Water resist